# 01. Auto Loader でファイルを取り込む

Auto Loader は、ストレージに置かれたファイルを**増分で**取り込む仕組みです。
一度読んだファイルを覚えていて、次に実行したときは新しく増えたファイルだけを読みます。

このノートブックで確かめること:

1. JSONファイルをDeltaテーブルに取り込む
2. 同じコードをもう一度実行すると何が起きるか
3. 「どこまで読んだか」をどうやって覚えているのか

**前提**: `00_setup` を実行して、カタログとVolumeを作ってあること。

## 準備

このノートブックはローカルのPythonで動きますが、Sparkの処理自体はDatabricks側で実行されます。
その橋渡しをするのが databricks-connect で、`DatabricksSession` がその入口です。

なお、VS CodeのDatabricks拡張機能を使っていると `spark` は自動で用意されるため、
本当はこのセルを書かなくても動きます。ここでは何が起きているかを見せるために明示的に作ります。

In [1]:
from databricks.connect import DatabricksSession

# serverless(True) = Databricks側のサーバーレスコンピュートに接続する
spark = DatabricksSession.builder.profile("free").serverless(True).getOrCreate()

In [2]:
# ファイル操作をDatabricks側に対して行うための道具
# ローカルから /Volumes/... を普通の open() で読み書きすることはできないため、これを経由する
from databricks.sdk.runtime import dbutils

CATALOG = "tech_survey"

# 取り込み先のテーブル
TABLE = f"{CATALOG}.bronze.orders_raw"

# 取り込み元のフォルダ - 他のノートブックとファイルが混ざらないよう、トピック名で分ける
LANDING = f"/Volumes/{CATALOG}/ops/landing/01_auto_loader"

# チェックポイント = Auto Loaderが「どこまで読んだか」を記録する場所 (後で詳しく見ます)
CHECKPOINT = f"/Volumes/{CATALOG}/ops/checkpoints/01_auto_loader"

## 1. 取り込み元のファイルを用意する

まず前回の実行結果を消します。こうしておくと、何度やり直しても同じ状態から始められます。
消すのはこのノートブック専用のフォルダとテーブルだけなので、他のトピックには影響しません。

In [3]:
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")

# 第2引数の True = フォルダの中身ごと消す
dbutils.fs.rm(LANDING, True)
dbutils.fs.rm(CHECKPOINT, True)

True

In [4]:
import json
import random
import uuid

# 注文データを50件作る
rows = []
for _ in range(50):
    rows.append(
        {
            "order_id": str(uuid.uuid4()),
            "product": random.choice(["laptop", "monitor", "keyboard"]),
            "amount": random.randint(1000, 50000),
        }
    )

# JSON Lines形式 = 1行に1レコード。Auto Loaderが読むのはこの形式
text = "\n".join(json.dumps(row) for row in rows)

dbutils.fs.put(f"{LANDING}/orders_1.json", text, True)

True

In [5]:
# 今フォルダに何があるか確認する
dbutils.fs.ls(LANDING)

[FileInfo(path='/Volumes/tech_survey/ops/landing/01_auto_loader/orders_1.json', name='orders_1.json', size=4597, modificationTime=1789200890000)]

## 2. Auto Loader で読む

`spark.read` ではなく **`spark.readStream`** を使います。この2つの違いが Auto Loader の肝です。

- `spark.read` … 毎回フォルダ全体を読む。前回何を読んだかは覚えていない
- `spark.readStream` … 前回の続きから読む。そのために「どこまで読んだか」を記録する

形式に `cloudFiles` を指定すると、この readStream が Auto Loader として動きます。

オプションの意味:

- `cloudFiles.format` … 読むファイルの形式。今回はJSON
- `cloudFiles.schemaLocation` … 推論した列名・型を保存しておく場所。
  JSONには型の情報がないので、Auto Loaderは中身を見て型を推測します。
  その結果をここに保存し、次回は読み直さずに再利用します

In [ ]:
# この時点ではまだ読み込みは始まりません。「こう読む」という定義を作っているだけです。
df = (
    spark.readStream.format("cloudFiles")  # `cloudFiles`で、Auto Loaderを使うことを宣言する
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT}/_schema")
    .load(LANDING)
)

## 3. テーブルに書き出す

読み取りの定義に、書き出し先を繋いで初めて処理が動きます。

**チェックポイント** がここで効いてきます。Auto Loader は処理したファイル名をチェックポイントに
記録します。次に同じチェックポイントを指定して実行すると、記録済みのファイルは飛ばして、
新しいファイルだけを読みます。これが「増分で取り込む」の中身です。
逆に言うと、チェックポイントを消せば最初から読み直しになります。

**トリガー** は処理をいつ動かすかの指定です。`availableNow=True` は
「今ある未処理のファイルを全部処理したら終了する」という意味で、
ノートブックでの確認や1日1回のバッチ処理に向いています。
指定しないと処理が終わらず、新しいファイルを待ち続けます。

In [ ]:
from pyspark.sql.streaming import StreamingQuery

query: StreamingQuery = (
    df.writeStream.option("checkpointLocation", CHECKPOINT)  # チェックポイントの場所を指定する
    .trigger(availableNow=True)
    .toTable(TABLE)
)

# 上の行を書いた時点で処理は走り出しています。終わるまで待つ
query.awaitTermination()

In [ ]:
display(spark.table(TABLE))

In [ ]:
spark.table(TABLE).count()

## 4. もう一度実行すると、どうなるか

ここが一番確かめたいところです。新しいファイルを1つ追加してから、
**さっきとまったく同じ取り込み処理** をもう一度動かします。

動かす前に予想してみてください。テーブルの件数はどうなるでしょうか。

- 追加した分だけ増える
- 最初のファイルも読み直されて倍近くになる

どちらになるか、そしてそれはなぜか。

In [ ]:
# 30件の新しいファイルを追加する
rows = []
for _ in range(30):
    rows.append(
        {
            "order_id": str(uuid.uuid4()),
            "product": random.choice(["laptop", "monitor", "keyboard"]),
            "amount": random.randint(1000, 50000),
        }
    )

dbutils.fs.put(f"{LANDING}/orders_2.json", "\n".join(json.dumps(r) for r in rows), True)
dbutils.fs.ls(LANDING)

In [ ]:
# さっきと同じ内容。チェックポイントも同じ場所を指している
df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT}/_schema")
    .load(LANDING)
)

query = (
    df.writeStream.option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .toTable(TABLE)
)
query.awaitTermination()

In [ ]:
spark.table(TABLE).count()

## 5. チェックポイントの中を見る

「どこまで読んだか」が実際にどう保存されているかを覗いてみます。

In [ ]:
dbutils.fs.ls(CHECKPOINT)

## 考えてみる

- `4.` の結果は予想どおりでしたか。違ったなら、どこが思い込みだったでしょうか
- チェックポイントのフォルダを消してからもう一度実行すると、件数はどうなるでしょうか
- 1日1回だけ動かすバッチ処理を作るとしたら、この仕組みで何が嬉しいでしょうか

## 後片付け

作ったものを消したいときに実行します。